# Model Evaluation
This notebook evaluates the trained HMM models on unseen test data from the data_unseen folder.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path for imports
sys.path.append('../src')

## 1. Load Trained Models
Load the HMM models and scaler that were trained in notebook 03.

In [ ]:
# Load the trained models and scaler
models = joblib.load("../models/hmm_models.pkl")
scaler = joblib.load("../models/scaler.pkl")

print("Loaded models for activities:", list(models.keys()))
print("Scaler loaded successfully")

## 2. Load Unseen Test Data
Load sensor data from the data_unseen directory and extract features.

In [ ]:
from feature_extraction import extract_features_from_recording

def load_unseen_data(data_unseen_dir='../data_unseen'):
    """
    Load and extract features from unseen test data.
    """
    all_features = []
    
    participants = ['ange', 'david']
    activities = ['standing', 'walking', 'jumping', 'still']
    
    for participant in participants:
        participant_dir = Path(data_unseen_dir) / participant
        
        if not participant_dir.exists():
            print(f"Warning: {participant_dir} not found")
            continue
            
        for activity in activities:
            activity_dir = participant_dir / activity
            
            if not activity_dir.exists():
                print(f"Warning: {activity_dir} not found")
                continue
            
            # Get all recording folders
            recordings = sorted([d for d in activity_dir.iterdir() if d.is_dir()])
            
            for recording in recordings:
                # Check for sensor files
                accel_file = recording / 'Accelerometer.csv'
                gyro_file = recording / 'Gyroscope.csv'
                
                if accel_file.exists() and gyro_file.exists():
                    try:
                        features = extract_features_from_recording(
                            str(accel_file),
                            str(gyro_file),
                            activity,
                            recording.name
                        )
                        all_features.append(features)
                    except Exception as e:
                        print(f"Error processing {recording.name}: {e}")
    
    return pd.DataFrame(all_features)

print("Loading unseen test data...")
test_data = load_unseen_data()
print(f"Loaded {len(test_data)} test samples")
print(f"Activities: {test_data['activity'].value_counts()}")

## 3. Preprocess Test Data
Apply the same preprocessing (scaling) used during training.

In [ ]:
# Separate features and labels
X_test = test_data.drop(columns=["activity", "recording"])
y_test = test_data["activity"]

# Apply the same scaling as training data
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"Test features shape: {X_test_scaled.shape}")
print(f"Test labels shape: {y_test.shape}")

## 4. Make Predictions
Use the trained HMM models to predict activities for the unseen test data.

In [ ]:
def predict_activity(sample, models):
    """
    Predict activity for a single sample using trained HMM models.
    Returns the activity with the highest likelihood score.
    """
    scores = {}
    
    for activity, model in models.items():
        try:
            score = model.score(sample)
            scores[activity] = score
        except:
            scores[activity] = -np.inf
    
    return max(scores, key=scores.get), scores

# Make predictions on test data
print("Making predictions on unseen test data...")
predictions = []
all_scores = []

for i in range(len(X_test_scaled)):
    sample = X_test_scaled.iloc[i:i+1]
    pred, scores = predict_activity(sample, models)
    predictions.append(pred)
    all_scores.append(scores)

predictions = np.array(predictions)
print("Predictions completed!")

## 5. Calculate Metrics
Evaluate model performance using various metrics.

In [ ]:
# Calculate overall accuracy
accuracy = accuracy_score(y_test, predictions)

print("="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, predictions))

## 6. Confusion Matrix
Visualize the confusion matrix to see which activities are confused with each other.

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, predictions)
activities = sorted(y_test.unique())

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=activities, yticklabels=activities,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Unseen Test Data', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Activity', fontsize=12)
plt.ylabel('True Activity', fontsize=12)
plt.tight_layout()

# Save figure
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/confusion_matrix_unseen.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to results/figures/")

## 7. Per-Activity Performance
Analyze performance for each activity individually.

In [ ]:
# Calculate per-activity accuracy
print("\n" + "="*60)
print("PER-ACTIVITY ACCURACY")
print("="*60)

for activity in activities:
    mask = y_test == activity
    activity_accuracy = accuracy_score(y_test[mask], predictions[mask])
    total_samples = mask.sum()
    correct_samples = (predictions[mask] == activity).sum()
    
    print(f"\n{activity.upper()}")
    print(f"  Accuracy: {activity_accuracy:.4f} ({activity_accuracy*100:.2f}%)")
    print(f"  Correct: {correct_samples}/{total_samples}")
    
    # Show most common misclassifications
    misclassified = predictions[mask & (predictions != activity)]
    if len(misclassified) > 0:
        from collections import Counter
        misclass_counts = Counter(misclassified)
        print(f"  Most common misclassifications:")
        for misclass, count in misclass_counts.most_common(2):
            print(f"    - {misclass}: {count} times")

## 8. Likelihood Score Analysis
Analyze the likelihood scores to understand model confidence.

In [ ]:
# Convert scores to DataFrame
scores_df = pd.DataFrame(all_scores)
scores_df['true_activity'] = y_test.values
scores_df['predicted_activity'] = predictions
scores_df['correct'] = (y_test.values == predictions)

# Calculate confidence (difference between top 2 scores)
def calculate_confidence(row):
    scores = [row[act] for act in activities]
    sorted_scores = sorted(scores, reverse=True)
    return sorted_scores[0] - sorted_scores[1]

scores_df['confidence'] = scores_df.apply(calculate_confidence, axis=1)

# Summary statistics
print("\n" + "="*60)
print("PREDICTION CONFIDENCE ANALYSIS")
print("="*60)
print(f"\nAverage confidence (correct predictions): {scores_df[scores_df['correct']]['confidence'].mean():.2f}")
print(f"Average confidence (incorrect predictions): {scores_df[~scores_df['correct']]['confidence'].mean():.2f}")

# Plot confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence for correct vs incorrect
axes[0].hist([scores_df[scores_df['correct']]['confidence'],
              scores_df[~scores_df['correct']]['confidence']],
             label=['Correct', 'Incorrect'], bins=20, alpha=0.7)
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Prediction Confidence Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confidence by activity
for activity in activities:
    mask = scores_df['true_activity'] == activity
    axes[1].hist(scores_df[mask]['confidence'], alpha=0.5, label=activity, bins=15)
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Confidence Distribution by Activity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Save Results
Save evaluation results to files for documentation.

In [ ]:
# Create results directory
os.makedirs('../results/metrics', exist_ok=True)
os.makedirs('../results/predictions', exist_ok=True)

# Save confusion matrix
cm_df = pd.DataFrame(cm, index=activities, columns=activities)
cm_df.to_csv('../results/metrics/confusion_matrix_unseen.csv')

# Save detailed predictions
predictions_df = pd.DataFrame({
    'recording': test_data['recording'].values,
    'true_activity': y_test.values,
    'predicted_activity': predictions,
    'correct': y_test.values == predictions
})

# Add likelihood scores for each activity
for activity in activities:
    predictions_df[f'score_{activity}'] = scores_df[activity].values
predictions_df['confidence'] = scores_df['confidence'].values

predictions_df.to_csv('../results/predictions/unseen_test_predictions.csv', index=False)

# Save evaluation metrics
with open('../results/metrics/evaluation_summary.txt', 'w') as f:
    f.write("="*60 + "\n")
    f.write("HMM ACTIVITY RECOGNITION - EVALUATION SUMMARY\n")
    f.write("="*60 + "\n\n")
    f.write(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)\n\n")
    f.write("="*60 + "\n")
    f.write("CLASSIFICATION REPORT\n")
    f.write("="*60 + "\n")
    f.write(classification_report(y_test, predictions))
    f.write("\n" + "="*60 + "\n")
    f.write("PER-ACTIVITY ACCURACY\n")
    f.write("="*60 + "\n")
    for activity in activities:
        mask = y_test == activity
        activity_accuracy = accuracy_score(y_test[mask], predictions[mask])
        f.write(f"\n{activity}: {activity_accuracy:.4f} ({activity_accuracy*100:.2f}%)\n")

print("\n" + "="*60)
print("Results saved to:")
print("  - results/metrics/confusion_matrix_unseen.csv")
print("  - results/predictions/unseen_test_predictions.csv")
print("  - results/metrics/evaluation_summary.txt")
print("  - results/figures/confusion_matrix_unseen.png")
print("  - results/figures/confidence_analysis.png")
print("="*60)

## Summary

This evaluation notebook:
1. ✅ Loaded trained HMM models and scaler
2. ✅ Loaded unseen test data from data_unseen/
3. ✅ Extracted features and applied same preprocessing
4. ✅ Made predictions using HMM likelihood scores
5. ✅ Calculated comprehensive metrics (accuracy, precision, recall, F1)
6. ✅ Created confusion matrix visualization
7. ✅ Analyzed per-activity performance
8. ✅ Examined prediction confidence
9. ✅ Saved all results for documentation

The model is now fully evaluated on unseen data!